# Ejercicio 7: Interpretación y Reporte de Clustering

Se debe realizar una interpretación y reporte de comunicación

1. Describir características comunes de cada cluster.
2. Generar visualizaciones (gráficos comparativos, mapas de calor, etc.).
3. Redactar informe con metodología, resultados y conclusiones.

**Informe completo disponible en:** `Docs/ejercicio7.md`

## Paso 1: Carga de datos y preparación


In [ ]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

# Cargar dataset
df = cargar_dataset_adult()
print(f"✓ Dataset cargado: {len(df):,} filas, {len(df.columns)} columnas")


## Paso 2: Clustering con K-means (Variables Numéricas)


In [ ]:
# Seleccionar variables numéricas (excluyendo fnlwgt)
numeric_cols = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

# Preparar datos numéricos
df_numeric = df[numeric_cols].copy()

# Normalizar
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_numeric)

# Aplicar K-means con k=4 (número óptimo determinado en Ejercicio 4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster_kmeans'] = kmeans.fit_predict(df_scaled)

# Métricas
silhouette_km = silhouette_score(df_scaled, df['cluster_kmeans'])
davies_bouldin_km = davies_bouldin_score(df_scaled, df['cluster_kmeans'])

print(f"✓ K-means aplicado (k=4)")
print(f"  Silhouette Score: {silhouette_km:.4f}")
print(f"  Davies-Bouldin Score: {davies_bouldin_km:.4f}")
print(f"\nDistribución de clusters:")
print(df['cluster_kmeans'].value_counts().sort_index())


## Paso 3: Análisis de características por cluster (K-means)


In [ ]:
# Análisis de características numéricas por cluster
cluster_stats = df.groupby('cluster_kmeans')[numeric_cols].agg(['mean', 'median', 'std'])
print("Estadísticas numéricas por cluster (K-means):")
print("=" * 80)
display(cluster_stats)

# Análisis de características categóricas
print("\n\nCaracterísticas categóricas principales por cluster:")
print("=" * 80)

categorical_features = ['workclass', 'education', 'marital-status', 'occupation', 'sex', 'income']

for cluster_id in sorted(df['cluster_kmeans'].unique()):
    cluster_data = df[df['cluster_kmeans'] == cluster_id]
    print(f"\n{'='*80}")
    print(f"CLUSTER {cluster_id} (n={len(cluster_data):,} personas - {len(cluster_data)/len(df)*100:.1f}%)")
    print(f"{'='*80}")
    
    for feature in categorical_features:
        top_values = cluster_data[feature].value_counts().head(3)
        print(f"\n{feature}:")
        for val, count in top_values.items():
            pct = count / len(cluster_data) * 100
            print(f"  • {val}: {count:,} ({pct:.1f}%)")


## Paso 4: Visualizaciones de clusters (K-means)


In [ ]:
# 1. Gráfico de barras comparativo - Variables numéricas
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Características Numéricas por Cluster (K-means)', fontsize=16, fontweight='bold')

for idx, col in enumerate(numeric_cols):
    ax = axes[idx // 3, idx % 3]
    cluster_means = df.groupby('cluster_kmeans')[col].mean()
    bars = ax.bar(cluster_means.index, cluster_means.values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    ax.set_xlabel('Cluster', fontsize=12)
    ax.set_ylabel(f'{col} (promedio)', fontsize=12)
    ax.set_title(col.replace('-', ' ').title(), fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Agregar valores en las barras
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}',
                ha='center', va='bottom', fontsize=10)

# Eliminar el subplot vacío
fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.show()


In [ ]:
# 2. Scatter plots - Relaciones entre variables clave
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Visualización de Clusters en Espacio 2D (K-means)', fontsize=16, fontweight='bold')

# Age vs Education
axes[0].scatter(df['age'], df['education-num'], c=df['cluster_kmeans'], 
                cmap='viridis', alpha=0.5, s=10)
axes[0].set_xlabel('Edad', fontsize=12)
axes[0].set_ylabel('Nivel Educativo', fontsize=12)
axes[0].set_title('Edad vs Educación', fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3)

# Hours-per-week vs Capital-gain
axes[1].scatter(df['hours-per-week'], df['capital-gain'], c=df['cluster_kmeans'], 
                cmap='viridis', alpha=0.5, s=10)
axes[1].set_xlabel('Horas de trabajo por semana', fontsize=12)
axes[1].set_ylabel('Ganancia de capital', fontsize=12)
axes[1].set_title('Horas trabajadas vs Ganancia de capital', fontsize=13, fontweight='bold')
axes[1].set_yscale('log')
axes[1].grid(alpha=0.3)

# Age vs Capital-gain
axes[2].scatter(df['age'], df['capital-gain'], c=df['cluster_kmeans'], 
                cmap='viridis', alpha=0.5, s=10)
axes[2].set_xlabel('Edad', fontsize=12)
axes[2].set_ylabel('Ganancia de capital', fontsize=12)
axes[2].set_title('Edad vs Ganancia de capital', fontsize=13, fontweight='bold')
axes[2].set_yscale('log')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# 3. Mapa de calor - Características normalizadas por cluster
cluster_means_normalized = df.groupby('cluster_kmeans')[numeric_cols].mean()

# Normalizar para mejor visualización (z-score)
from sklearn.preprocessing import StandardScaler
scaler_viz = StandardScaler()
cluster_means_scaled = scaler_viz.fit_transform(cluster_means_normalized)
cluster_means_scaled_df = pd.DataFrame(
    cluster_means_scaled, 
    index=[f'Cluster {i}' for i in cluster_means_normalized.index],
    columns=[col.replace('-', ' ').title() for col in numeric_cols]
)

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_means_scaled_df.T, annot=True, fmt='.2f', cmap='RdYlGn', 
            center=0, linewidths=1, cbar_kws={'label': 'Desviaciones estándar'})
plt.title('Mapa de Calor: Características Normalizadas por Cluster (K-means)', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Clusters', fontsize=12)
plt.ylabel('Variables', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# 4. Distribución de variables categóricas clave por cluster
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Distribución de Variables Categóricas por Cluster (K-means)', 
             fontsize=16, fontweight='bold')

# Income
income_counts = pd.crosstab(df['cluster_kmeans'], df['income'], normalize='index') * 100
income_counts.plot(kind='bar', stacked=True, ax=axes[0,0], color=['#e74c3c', '#2ecc71'])
axes[0,0].set_title('Distribución de Ingresos', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Cluster', fontsize=11)
axes[0,0].set_ylabel('Porcentaje (%)', fontsize=11)
axes[0,0].legend(title='Income', loc='best')
axes[0,0].set_xticklabels(axes[0,0].get_xticklabels(), rotation=0)

# Sex
sex_counts = pd.crosstab(df['cluster_kmeans'], df['sex'], normalize='index') * 100
sex_counts.plot(kind='bar', stacked=True, ax=axes[0,1], color=['#3498db', '#e91e63'])
axes[0,1].set_title('Distribución por Sexo', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('Cluster', fontsize=11)
axes[0,1].set_ylabel('Porcentaje (%)', fontsize=11)
axes[0,1].legend(title='Sex', loc='best')
axes[0,1].set_xticklabels(axes[0,1].get_xticklabels(), rotation=0)

# Workclass (top 5)
top_workclass = df['workclass'].value_counts().head(5).index
df_work_filtered = df[df['workclass'].isin(top_workclass)]
work_counts = pd.crosstab(df_work_filtered['cluster_kmeans'], df_work_filtered['workclass'], 
                          normalize='index') * 100
work_counts.plot(kind='bar', ax=axes[1,0])
axes[1,0].set_title('Tipo de Trabajo (Top 5)', fontsize=13, fontweight='bold')
axes[1,0].set_xlabel('Cluster', fontsize=11)
axes[1,0].set_ylabel('Porcentaje (%)', fontsize=11)
axes[1,0].legend(title='Workclass', loc='best', fontsize=8)
axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(), rotation=0)

# Marital Status (top 5)
top_marital = df['marital-status'].value_counts().head(5).index
df_marital_filtered = df[df['marital-status'].isin(top_marital)]
marital_counts = pd.crosstab(df_marital_filtered['cluster_kmeans'], 
                             df_marital_filtered['marital-status'], normalize='index') * 100
marital_counts.plot(kind='bar', ax=axes[1,1])
axes[1,1].set_title('Estado Civil (Top 5)', fontsize=13, fontweight='bold')
axes[1,1].set_xlabel('Cluster', fontsize=11)
axes[1,1].set_ylabel('Porcentaje (%)', fontsize=11)
axes[1,1].legend(title='Marital Status', loc='best', fontsize=8)
axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()
